## 02 — Sentiment volatility

**Reproducible run:** requires `data/raw/helpdesk_tickets.csv` from `python scripts/generate_synthetic_data.py`. This notebook uses an in-memory SQLite engine as a local stand-in for the warehouse (no external SQL Server needed); the production Postgres/T-SQL lives in `SQL/`. Data is synthetic.

# Sentiment Volatility Analysis
GitHub Support Operations Analytics - Sentiment Shift Detection

This notebook identifies the "Frustration Metric" by analyzing sentiment delta between initial issue and final comment.
Helps distinguish between Product Problems vs Support Problems.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import warnings
warnings.filterwarnings('ignore')


nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/ccuser/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [2]:
# Local, reproducible source: the shipped synthetic CSV + an
# in-memory SQLite engine (no external SQL Server needed).
import sqlalchemy
engine = sqlalchemy.create_engine('sqlite://')  # in-memory
src = pd.read_csv('../data/raw/helpdesk_tickets.csv')
src = src.rename(columns={'Ticket_Key': 'Ticket_Key',
                          'Issue_Question': 'Issue_Question',
                          'Last_Comment_Text': 'Last_Comment_Text'})
src[['Ticket_Key', 'Issue_Question', 'Last_Comment_Text']]\
    .to_sql('Fact_Tickets', engine, if_exists='replace', index=False)


2000

In [3]:
# 2. Fetch the raw text to score
query = "SELECT Ticket_Key, Issue_Question, Last_Comment_Text FROM Fact_Tickets"
df = pd.read_sql(query, engine)

print(f"Loaded {len(df)} tickets for sentiment analysis")
print(df.head())


Loaded 2000 tickets for sentiment analysis
  Ticket_Key                                     Issue_Question  \
0    T-00001  Feature request: would love to see dark mode i...   
1    T-00002  Critical: SSRF vulnerability allows unauthoriz...   
2    T-00003  Feature request: would love to see dark mode i...   
3    T-00004  Build fails after upgrading to v4.6 with error...   
4    T-00005   How do I configure OAuth2? The docs are unclear.   

                              Last_Comment_Text  
0     Unfortunately this is by design. Closing.  
1  Can you provide a minimal reproduction case?  
2  Thanks for the report! We'll look into this.  
3  Thanks for the report! We'll look into this.  
4      This is a known issue, tracking in #366.  


In [4]:
# 3. Sentiment Analysis Function
def get_score(text):
    if not text or pd.isna(text):
        return 0
    return sia.polarity_scores(str(text))['compound']

In [5]:

df['Initial_Sentiment'] = df['Issue_Question'].apply(get_score)
df['Final_Sentiment'] = df['Last_Comment_Text'].apply(get_score)

print("Sentiment scores calculated:")
print(df[['Initial_Sentiment', 'Final_Sentiment']].describe())

Sentiment scores calculated:
       Initial_Sentiment  Final_Sentiment
count        2000.000000      2000.000000
mean           -0.208939         0.069050
std             0.393217         0.271771
min            -0.670500        -0.726900
25%            -0.493900         0.000000
50%            -0.296000         0.000000
75%             0.000000         0.318200
max             0.636900         0.492600


In [6]:
# 5. Calculate Sentiment Volatility (Shift)
# Negative shift means user got angrier/more frustrated during the process
df['Sentiment_Shift'] = df['Final_Sentiment'] - df['Initial_Sentiment']

# "Toxic Interaction" flag: Initial sentiment was neutral/positive, but final was negative
df['is_frustrated_user'] = ((df['Initial_Sentiment'] >= 0) & (df['Sentiment_Shift'] < -0.5)).astype(int)

print("\nSentiment Shift Statistics:")
print(df['Sentiment_Shift'].describe())
print(f"\nFrustrated Users Count: {df['is_frustrated_user'].sum()}")


Sentiment Shift Statistics:
count    2000.000000
mean        0.277989
std         0.484615
min        -1.363800
25%         0.000000
50%         0.318200
75%         0.636400
max         1.163100
Name: Sentiment_Shift, dtype: float64

Frustrated Users Count: 178


In [7]:
# 6. Upload to SQL as a new Sentiment Fact Table
sentiment_output = df[['Ticket_Key', 'Initial_Sentiment', 'Final_Sentiment', 'Sentiment_Shift', 'is_frustrated_user']]

sentiment_output.to_sql(
    'Fact_Sentiment', 
    con=engine, 
    if_exists='replace', 
    index=False
)

print("Phase 2 Complete: Sentiment Data pushed to Fact_Sentiment table.")
print(f"Total records: {len(sentiment_output)}")
print(f"Frustrated users flagged: {sentiment_output['is_frustrated_user'].sum()}")

Phase 2 Complete: Sentiment Data pushed to Fact_Sentiment table.
Total records: 2000
Frustrated users flagged: 178


In [8]:
# 7.  Statistics
print("\n=== SENTIMENT VOLATILITY SUMMARY ===")
print(f"Average Initial Sentiment: {df['Initial_Sentiment'].mean():.3f}")
print(f"Average Final Sentiment: {df['Final_Sentiment'].mean():.3f}")
print(f"Average Sentiment Shift: {df['Sentiment_Shift'].mean():.3f}")
print(f"\nPositive Shift (User Got Happier): {(df['Sentiment_Shift'] > 0.1).sum()} tickets")
print(f"Negative Shift (User Got Frustrated): {(df['Sentiment_Shift'] < -0.1).sum()} tickets")
print(f"Stable Sentiment: {((df['Sentiment_Shift'] >= -0.1) & (df['Sentiment_Shift'] <= 0.1)).sum()} tickets")


=== SENTIMENT VOLATILITY SUMMARY ===
Average Initial Sentiment: -0.209
Average Final Sentiment: 0.069
Average Sentiment Shift: 0.278

Positive Shift (User Got Happier): 1402 tickets
Negative Shift (User Got Frustrated): 364 tickets
Stable Sentiment: 234 tickets
